In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from tqdm import tqdm
import earthaccess
import warnings

warnings.filterwarnings("ignore")

# =========================
# USER CONFIG
# =========================
START_DATE = "2020-01-01"
END_DATE   = "2025-12-31"

bbox = (-160.5, 19.0, -155.5, 24.0)  # (west, south, east, north)
SHORT_NAME = "OSTIA-UKMO-L4-GLOB-v2.0"

OUT_ROOT = Path("Y:/") / "Mingyue" / "Oahu" / "L4_OSTIA_Oahu"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

NODATA = -9999.0

# =========================
# HELPERS
# =========================
def _maybe_wrap_lon(ds: xr.Dataset, w: float, e: float):
    """
    If ds lon is 0..360 but bbox is -180..180, convert bbox to 0..360.
    Returns (w2,e2) for slicing.
    """
    lon = ds["lon"].values
    lon_min = float(np.nanmin(lon))
    lon_max = float(np.nanmax(lon))

    # Dataset in [0, 360] and bbox looks like negative longitudes
    if lon_min >= 0 and lon_max > 180 and (w < 0 or e < 0):
        w2 = (w + 360) % 360
        e2 = (e + 360) % 360
        return w2, e2

    return w, e


def write_geotiff(sst: xr.DataArray, out_tif: Path):
    """Exports a 2D xarray DataArray to a GeoTIFF (EPSG:4326)."""
    lats = sst["lat"].values
    lons = sst["lon"].values

    # assume regular grid
    dlat = float(np.abs(lats[1] - lats[0]))
    dlon = float(np.abs(lons[1] - lons[0]))

    north = float(np.nanmax(lats))
    west  = float(np.nanmin(lons))

    transform = from_origin(west, north, dlon, dlat)

    data = sst.values.astype(np.float32)

    # Ensure north-up row order for rasterio
    if lats[0] < lats[-1]:
        data = np.flipud(data)

    # Replace NaNs with numeric nodata for clean GeoTIFF metadata
    data_out = np.where(np.isfinite(data), data, np.float32(NODATA)).astype(np.float32)

    out_tif.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=data_out.shape[0],
        width=data_out.shape[1],
        count=1,
        dtype="float32",
        crs="EPSG:4326",
        transform=transform,
        nodata=np.float32(NODATA),
        compress="deflate",
        predictor=2,
    ) as dst:
        dst.write(data_out, 1)


def safe_ymd_from_name(name: str) -> str:
    # Common case: YYYYMMDDhhmmss-...
    if len(name) >= 8 and name[:8].isdigit():
        return name[:8]
    # fallback: try to find an 8-digit token
    import re
    m = re.search(r"(\d{8})", name)
    if m:
        return m.group(1)
    return "unknown_date"


# =========================
# MAIN
# =========================
def main():
    print("Authenticating with NASA Earthdata...")
    earthaccess.login(strategy="interactive")

    print(f"\nSearching for {SHORT_NAME} granules...")
    results = earthaccess.search_data(
        short_name=SHORT_NAME,
        temporal=(START_DATE, END_DATE),
        bounding_box=bbox,
        cloud_hosted=True
    )

    if not results:
        print("No data found for this time range and bounding box.")
        return

    print(f"Found {len(results)} files. Starting virtual extraction...\n")

    w, s, e, n = bbox

    for granule in tqdm(results, desc="Processing OSTIA L4"):
        # safer link/name extraction
        links = granule.data_links(access="direct")
        granule_name = (links[0].split("/")[-1] if links else (granule.title or "unknown"))
        ymd = safe_ymd_from_name(granule_name)

        out_day_dir = OUT_ROOT / "geotiff" / ymd[:4] / ymd[4:6]
        out_tif = out_day_dir / f"SST_L4_OSTIA_{ymd}.tif"
        if out_tif.exists():
            continue

        try:
            f_obj = earthaccess.open([granule])[0]

            with xr.open_dataset(f_obj, engine="h5netcdf") as ds:
                # handle lon convention
                w2, e2 = _maybe_wrap_lon(ds, w, e)

                # robust slicing even if coords are descending
                lat_vals = ds["lat"].values
                lon_vals = ds["lon"].values

                lat_slice = slice(s, n) if lat_vals[0] < lat_vals[-1] else slice(n, s)

                # lon may be ascending; if descending, invert slice
                lon_slice = slice(w2, e2) if lon_vals[0] < lon_vals[-1] else slice(e2, w2)

                ds_sub = ds.sel(lon=lon_slice, lat=lat_slice)

                if ds_sub.dims.get("lat", 0) == 0 or ds_sub.dims.get("lon", 0) == 0:
                    raise ValueError("Subset is empty (check bbox vs dataset lon/lat convention).")

                sst = ds_sub["analysed_sst"].squeeze()

                # Kelvin -> Celsius + basic sanity filter
                sst_c = (sst - 273.15).where(lambda x: (x > 0) & (x < 40))

                write_geotiff(sst_c, out_tif)

        except Exception as err:
            print(f"\n[WARN] Failed to process {ymd} ({granule_name}): {err}")

    print("\nDONE. GeoTIFFs saved to:", OUT_ROOT / "geotiff")


if _name_ == "_main_":
    main()